In [ ]:
pip install Pypdf2 pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 80.2 MB/s eta 0:00:00


In [25]:
# =========================================================
# RESUME PARSER — FULL WORKING COLAB CODE
# =========================================================

import json
import re
import time
import google.generativeai as genai
import typing_extensions as typing
from google.colab import userdata
from PyPDF2 import PdfReader



genai.configure(
    api_key=userdata.get("GEMINI_API_KEYS")
)



model = genai.GenerativeModel(
    "gemini-2.5-flash-lite",
    system_instruction="""
You are an advanced resume parser.

RULES:
1. Extract only information explicitly present in resume.
2. Never invent or hallucinate values.
3. Use null if missing.
4. Output MUST be valid JSON only.
5. Normalize phone to +91XXXXXXXXXX.
6. Skills must be unique and clean.
7. Summary must be one sentence.
8. Resume may be in any Indian language but output must be English.
"""
)

# =========================================================
# SCHEMA (FIXED AND CONSISTENT)
# =========================================================

class Education(typing.TypedDict):
    degree: str | None
    institution: str | None
    year: int | None
    cgpa: str | None
    percentage: str | None


class Experience(typing.TypedDict):
    title: str | None
    company: str | None
    start_date: str | None
    end_date: str | None
    description: str | None


class Project(typing.TypedDict):
    name: str | None
    description: str | None
    tech_stack: list[str]


class Resume(typing.TypedDict):
    name: str | None
    email: str | None
    phone: str | None
    linkedin_url: str | None

    education: list[Education]
    skills: list[str]
    experience: list[Experience]
    projects: list[Project]

    total_experience_years: float
    summary: str | None

# =========================================================
# EXPECTED KEYS
# =========================================================

EXPECTED_KEYS = [
    "name",
    "email",
    "phone",
    "linkedin_url",
    "education",
    "skills",
    "experience",
    "projects",
    "total_experience_years",
    "summary"
]

# =========================================================
# SAFE JSON PARSER
# =========================================================

def safe_json_parse(text):
    try:
        text = text.strip()
        text = text.replace("```json", "").replace("```", "")

        start = text.find("{")
        end = text.rfind("}")

        if start != -1 and end != -1:
            text = text[start:end + 1]

        return json.loads(text)

    except Exception as e:
        print("JSON Parse Error:", e)
        return None

# =========================================================
# VALIDATION
# =========================================================

def validate_data(data):
    if not data:
        data = {}

    cleaned = {}
    for k in EXPECTED_KEYS:
        cleaned[k] = data.get(k, None)

    data = cleaned

    # email check
    if data["email"] and "@" not in data["email"]:
        data["email"] = None

    # phone normalize
    if data["phone"]:
        digits = re.sub(r"\D", "", str(data["phone"]))
        if len(digits) >= 10:
            data["phone"] = "+91" + digits[-10:]
        else:
            data["phone"] = None

    # linkedin normalize
    if data["linkedin_url"] and not data["linkedin_url"].startswith("http"):
        data["linkedin_url"] = "https://" + data["linkedin_url"]

    # experience validation
    try:
        data["total_experience_years"] = float(data["total_experience_years"])
        if data["total_experience_years"] < 0:
            data["total_experience_years"] = 0.0
    except:
        data["total_experience_years"] = 0.0

    # skills cleanup
    if data["skills"]:
        unique = []
        for s in data["skills"]:
            s = s.strip()
            if s not in unique:
                unique.append(s)
        data["skills"] = unique

    return data

# =========================================================
# PARSER FUNCTION
# =========================================================

def parse_resume(text):

    prompt = f"""
Extract resume details.

Return ONLY JSON.

Keys:
name, email, phone, linkedin_url,
education, skills, experience, projects,
total_experience_years, summary

Resume:
{text}
"""

    time.sleep(2)

    # ================= WAY 1 =================
    print("\n================ WAY 1 =================")

    r1 = model.generate_content(prompt)
    print(r1.text)

    try:
        d1 = safe_json_parse(r1.text)
        print("WAY 1 JSON OK")
    except:
        print("WAY 1 FAILED")

    # ================= WAY 2 =================
    print("\n================ WAY 2 =================")

    r2 = model.generate_content(
        prompt,
        generation_config={
            "response_mime_type": "application/json",
            "temperature": 0.2,
            "max_output_tokens": 2048
        }
    )

    d2 = safe_json_parse(r2.text)
    d2 = validate_data(d2)

    print(json.dumps(d2, indent=2, ensure_ascii=False))

    # ================= WAY 3 =================
    print("\n================ WAY 3 =================")

    r3 = model.generate_content(
        prompt,
        generation_config={
            "response_mime_type": "application/json",
            "response_schema": Resume,
            "temperature": 0.2,
            "max_output_tokens": 2048
        }
    )

    d3 = safe_json_parse(r3.text)
    d3 = validate_data(d3)

    print(json.dumps(d3, indent=2, ensure_ascii=False))

    # ================= SUMMARY =================
    print("\n================ SUMMARY =================")

    print("Name:", d3.get("name"))
    print("Email:", d3.get("email"))
    print("Phone:", d3.get("phone"))
    print("Skills:", d3.get("skills"))
    print("Experience:", d3.get("total_experience_years"))

    return d3

# =========================================================
# TEST 1
# =========================================================

resume_1 = """
ANJALI SHARMA
Email: anjali.sharma@gmail.com | Phone: 9876543210
LinkedIn: linkedin.com/in/anjali-sharma

Skills: Python, Django, React

Education:
B.Tech CSE IIT Delhi 2024 CGPA 8.9

Experience:
Intern at Flipkart

Projects:
AgriBot using Gemini API
"""

print("\n################ TEST 1 ################")
parse_resume(resume_1)

# =========================================================
# TEST 2
# =========================================================

resume_2 = """
hii i am ravi (ravi.k@yahoo.in / 8765-432-100)
btech vit vellore 2023 cgpa 8
java python c++
intern at zoho 6 months
backend dev at bytemate 8 months
"""

print("\n################ TEST 2 ################")
parse_resume(resume_2)

# =========================================================
# TEST 3 — PDF
# =========================================================

print("\n################ TEST 3 PDF ################")

pdf_path = "/content/resumebest.pdf"

try:
    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t

    parse_resume(text)

except Exception as e:
    print("PDF ERROR:", e)


################ TEST 1 ################

================ WAY 1 =================
```json
{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin_url": "https://linkedin.com/in/anjali-sharma",
  "education": [
    {
      "degree": "B.Tech",
      "major": "CSE",
      "institution": "IIT Delhi",
      "graduation_year": 2024,
      "cgpa": 8.9
    }
  ],
  "skills": [
    "Python",
    "Django",
    "React"
  ],
  "experience": [
    {
      "title": "Intern",
      "company": "Flipkart",
      "duration": null,
      "description": null
    }
  ],
  "projects": [
    {
      "name": "AgriBot using Gemini API",
      "description": null
    }
  ],
  "total_experience_years": null,
  "summary": null
}
```
WAY 1 JSON OK

================ WAY 2 =================
{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin_url": "https://linkedin.com/in/anjali-sharma",
  "education": [
    {

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1014.02ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1092.36ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1066.40ms


```json
{
  "name": "Ravi",
  "email": "ravi.k@yahoo.in",
  "phone": "+918765432100",
  "linkedin_url": null,
  "education": [
    {
      "degree": "B.Tech",
      "university": "VIT Vellore",
      "year": 2023,
      "cgpa": 8.0
    }
  ],
  "skills": [
    "Java",
    "Python",
    "C++"
  ],
  "experience": [
    {
      "title": "Intern",
      "company": "Zoho",
      "duration_months": 6
    },
    {
      "title": "Backend Developer",
      "company": "Bytemate",
      "duration_months": 8
    }
  ],
  "projects": [],
  "total_experience_years": 1.17,
  "summary": "Experienced backend developer with skills in Java, Python, and C++."
}
```
WAY 1 JSON OK

================ WAY 2 =================
{
  "name": "ravi",
  "email": "ravi.k@yahoo.in",
  "phone": "+918765432100",
  "linkedin_url": null,
  "education": [
    {
      "degree": "B.Tech",
      "university": "VIT Vellore",
      "year": 2023,
      "cgpa": "8"
    }
  ],
  "skills": [
    "Java",
    "Python",
    "C++"
  ]